En la siguiente parte uso el comando df = spark.table("workspace.default.retail_store_sales").toPandas() que según la documentación oficial de spark es una instancia de pyspark.sql.SparkSession y toPandas() es un método de pyspark.sql.DataFrame

Lectura de datos:

In [0]:
import re
import pandas as pd

df = spark.table("workspace.default.retail_store_sales").toPandas()

Buscando valores nulos, filas duplicadas

In [0]:
# ── Validaciones de formato ─────────────────────────
col1 = df.columns[0]  # Transaction ID
col2 = df.columns[1]  # Customer ID
col3 = df.columns[2]  # Category
col4 = df.columns[3]  # Item
col5 = df.columns[4]  # Price Per Unit
col6 = df.columns[5]  # Quantity
col7 = df.columns[6]  # Total Spent
col8 = df.columns[7]  # Payment Method
col9 = df.columns[8]  # Location
col10 = df.columns[9] # Date
col11 = df.columns[10] # Discount Applied

resultados[col1] = df[col1].astype(str).str.match(r'^TXN_\d{7}$')
resultados[col2] = df[col2].astype(str).str.match(r'^CUST_\d{2}$')
resultados[col3] = df[col3].notna()
resultados[col4] = df[col4].notna()
resultados[col5] = pd.to_numeric(df[col5], errors='coerce').notna()
resultados[col6] = pd.to_numeric(df[col6], errors='coerce').notna()
resultados[col7] = pd.to_numeric(df[col7], errors='coerce').notna()
resultados[col8] = df[col8].isin({'Credit Card', 'Digital Wallet', 'Cash'})
resultados[col9] = df[col9].isin({'Online', 'In-store'})
resultados[col10] = df[col10].astype(str).str.match(r'^\d{4}/\d{2}/\d{2}$')
resultados[col11] = df[col11].isin([True, False, 'True', 'False'])

Mostrando las columnas con errores:

In [0]:
# ── 1. Celdas vacías por columna ───────────────────
print("\n" + "=" * 60)
print("1. VALORES NULOS / CELDAS VACÍAS")
print("=" * 60)
nulos = df.isnull().sum()
pct   = df.isnull().mean() * 100
hay_nulos = False
for col in df.columns:
    if nulos[col] > 0:
        hay_nulos = True
        print(f"  {col:<25} {nulos[col]:>6} nulos  ({pct[col]:.1f}%)")
if not hay_nulos:
    print("  ✓ No se encontraron valores nulos")

# ── 2. Duplicados ──────────────────────────────────
print("\n" + "=" * 60)
print("2. DUPLICADOS")
print("=" * 60)

# Filas completamente duplicadas
dup_filas = df.duplicated().sum()
print(f"  Filas completamente duplicadas: {dup_filas}")

# Transaction ID duplicados (no deben repetirse)
dup_txn = df[col1].duplicated().sum()
print(f"  Transaction ID duplicados:      {dup_txn}")
if dup_txn > 0:
    print("\n  IDs duplicados encontrados:")
    ids_dup = df[df[col1].duplicated(keep=False)][[col1]]
    print(ids_dup.to_string())

# ── 3. Errores de formato por columna ─────────────
print("\n" + "=" * 60)
print("3. ERRORES DE FORMATO POR COLUMNA")
print("=" * 60)
columnas_excluidas = {col3, col4, col10}  # columnas de texto libre sin formato estricto y de fecha validada por aparte
total_errores = 0
for col, mascara in resultados.items():
    if col in columnas_excluidas:
        continue
    errores = (~mascara).sum()
    total_errores += errores
    estado = "✓ Sin errores" if errores == 0 else f"✗ {errores} registros inválidos ({errores/len(df)*100:.1f}%)"
    print(f"  {col:<25} {estado}")

# ── Validación extra: formato de fecha YYYY-MM-DD ──
fecha_valida = df[col10].astype(str).str.match(r'^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])$')
errores_fecha = (~fecha_valida).sum()
print(f"\n  Validación fecha (YYYY-MM-DD):")
print(f"  {col10:<25} {'✓ Sin errores' if errores_fecha == 0 else f'✗ {errores_fecha} fechas con formato inválido ({errores_fecha/len(df)*100:.1f}%)'}")
if errores_fecha > 0:
    print("\n  Ejemplos de fechas inválidas:")
    print(df.loc[~fecha_valida, col10].head(5).to_string())

print(f"\n  Total errores de formato: {total_errores}")

# ── 4. Detalle de errores por columna ─────────────
print("\n" + "=" * 60)
print("4. DETALLE DE VALORES INVÁLIDOS POR COLUMNA")
print("=" * 60)
for col, mascara in resultados.items():
    filas_error = df[~mascara]
    if len(filas_error) > 0:
        print(f"\n  ── {col} ({len(filas_error)} errores) ──")
        print(f"  Valores únicos inválidos encontrados:")
        print(f"  {df[~mascara][col].unique()}")
        
# ── 5. Categorías únicas para validar si alguna incumple el estandar explicito de los nombres de las columnas ───────────────────────────────
print("===== DISTRIBUCIÓN DE CATEGORÍAS =====\n")
print(df['Category'].value_counts())
print(f"\nTotal categorías únicas: {df['Category'].nunique()}")